# LPIPS Ranking Inference
Campiona N immagini dal val set, fa inferenza con il modello caricato,
salva le immagini generate su disco e ordina tutto per LPIPS.
Le celle di display possono essere ri-eseguite senza ripetere l'inferenza.

In [ ]:
!git clone https://github.com/Alessandro-Carotenuto/synt_satellite_vqgan_from_TT
%cd synt_satellite_vqgan_from_TT
!python setup.py

In [ ]:
# ─── USER CONFIGURATION ──────────────────────────────────────────────────────
CHECKPOINT_PATH = "/kaggle/input/models/username/model/pytorch/default/1/checkpoint.pth"  # <── CAMBIA QUI
DATA_ROOT       = "/kaggle/input/datasets/carotenutoalessandro/cvusa-groundandpolar-subset-35191"

N_SAMPLES    = 1000
RESULTS_DIR  = "lpips_ranking_results"   # cartella dove salvare immagini e JSON
TEMPERATURE  = 0.7
TOP_K        = 10
TOP_P        = 0.85
RANDOM_SEED  = 42
# ─────────────────────────────────────────────────────────────────────────────

## Setup

In [ ]:
import os, sys, json, random, csv
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# fixermodule VA importato per primo: al caricamento patcha su disco
# taming-transformers/taming/data/utils.py (torch._six fix) e inietta
# top_k_top_p_filtering in transformers. Solo dopo è sicuro importare taming_interface.
import fixermodule  # noqa: F401

import config as cfg
cfg.DATA_ROOT = DATA_ROOT  # override con il valore della cella di config

from taming_interface import load_saved_model
from CVUSA_Manager import get_standard_transform
from fixermodule import top_k_top_p_filtering
from taming.modules.losses.lpips import LPIPS

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
model, checkpoint, DEVICE = load_saved_model(CHECKPOINT_PATH)
model.eval()
print(f"Loaded — epoch {checkpoint['epoch']}, loss {checkpoint['loss']:.4f}")

_lpips_fn = LPIPS().to(DEVICE)
_lpips_fn.eval()
print("LPIPS ready")

## Campionamento dataset

In [ ]:
random.seed(RANDOM_SEED)

csv_path = os.path.join(DATA_ROOT,
    "val-19zl_fixed.csv" if cfg.OLD_SUBSET else "val.csv")

all_pairs = []
with open(csv_path, "r", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        if len(row) < 2:
            continue
        p = os.path.join(DATA_ROOT, row[0].replace("\\", os.sep).strip())
        g = os.path.join(DATA_ROOT, row[1].replace("\\", os.sep).strip())
        if os.path.exists(p) and os.path.exists(g):
            all_pairs.append((g, p))   # (ground_path, sat_path)

print(f"Coppie valide nel val set: {len(all_pairs)}")

# Salva gli indici campionati per riproducibilità
idx_file = os.path.join(RESULTS_DIR, "sample_indices.json")
if os.path.exists(idx_file):
    with open(idx_file) as f:
        sampled_idx = json.load(f)
    print(f"Indici campione caricati da disco ({len(sampled_idx)} coppie)")
else:
    n = min(N_SAMPLES, len(all_pairs))
    sampled_idx = random.sample(range(len(all_pairs)), n)
    with open(idx_file, "w") as f:
        json.dump(sampled_idx, f)
    print(f"Campionati {len(sampled_idx)} indici (seed={RANDOM_SEED})")

sampled_pairs = [all_pairs[i] for i in sampled_idx]
print(f"Pronti: {len(sampled_pairs)} coppie")

## Inferenza e salvataggio su disco
Se `results.json` esiste già, l'inferenza viene saltata (o ripresa dal punto di interruzione).
Per ripartire da zero cancella `lpips_ranking_results/`.

In [ ]:
_transform = get_standard_transform()
_to_pil = transforms.ToPILImage()

@torch.no_grad()
def _infer_one(ground_path):
    ground_tensor = _transform(
        Image.open(ground_path).convert("RGB")
    ).unsqueeze(0).to(DEVICE)

    _, _, info = model.cond_stage_model.encode(ground_tensor)
    cond_idx = info[2]
    B = ground_tensor.shape[0]
    if cond_idx.dim() == 1:
        cond_idx = cond_idx.view(B, -1)

    sequence = cond_idx
    for i in range(256):
        logits, _ = model.transformer(sequence)
        next_token_logits = logits[:, -1, :] / TEMPERATURE
        filtered_logits = top_k_top_p_filtering(next_token_logits, top_k=TOP_K, top_p=TOP_P)
        probs = torch.softmax(filtered_logits, dim=-1)
        next_token = torch.multinomial(probs, 1)
        sequence = torch.cat([sequence, next_token], dim=1)
    generated_tokens = sequence[:, -256:]

    tokens = generated_tokens.view(B, 16, 16)
    quant_z = model.first_stage_model.quantize.embedding(tokens)
    quant_z = quant_z.permute(0, 3, 1, 2).contiguous()
    return model.first_stage_model.decode(quant_z)

In [ ]:
results_file = os.path.join(RESULTS_DIR, "results.json")

# Carica risultati già presenti (supporta ripresa dopo crash)
if os.path.exists(results_file):
    with open(results_file) as f:
        results = json.load(f)
    print(f"Caricati {len(results)} risultati da cache")
else:
    results = []

done = {r["ground_path"] for r in results}
pending = [(g, s) for g, s in sampled_pairs if g not in done]
print(f"{len(pending)} campioni ancora da processare")

for ground_path, sat_path in tqdm(pending, desc="Inferenza"):
    try:
        generated = _infer_one(ground_path)

        real_tensor = _transform(
            Image.open(sat_path).convert("RGB")
        ).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            lpips_score = _lpips_fn(generated, real_tensor).item()

        stem = os.path.splitext(os.path.basename(ground_path))[0]
        gen_path = os.path.join(RESULTS_DIR, stem + "_gen.png")
        _to_pil(((generated.squeeze(0).cpu() + 1) / 2).clamp(0, 1)).save(gen_path)

        results.append({
            "ground_path": ground_path,
            "sat_path":    sat_path,
            "gen_path":    gen_path,
            "lpips":       lpips_score,
        })

        # Flush ad ogni sample: crash-safe
        with open(results_file, "w") as f:
            json.dump(results, f)

    except Exception as e:
        print(f"  Errore su {os.path.basename(ground_path)}: {e}")

sorted_results = sorted(results, key=lambda r: r["lpips"])
vals = [r["lpips"] for r in sorted_results]
print(f"\nTotale: {len(sorted_results)} campioni")
print(f"LPIPS  min={min(vals):.4f}  max={max(vals):.4f}  mean={np.mean(vals):.4f}  std={np.std(vals):.4f}")

In [ ]:
# Distribuzione LPIPS
vals = [r["lpips"] for r in sorted_results]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(vals, bins=50, color="steelblue", edgecolor="white")
ax.axvline(np.mean(vals), color="red", linestyle="--", label=f"mean={np.mean(vals):.4f}")
ax.axvline(np.median(vals), color="orange", linestyle="--", label=f"median={np.median(vals):.4f}")
ax.set_xlabel("LPIPS")
ax.set_ylabel("Conteggio")
ax.set_title(f"Distribuzione LPIPS su {len(vals)} campioni")
ax.legend()
plt.tight_layout()
plt.show()

## Visualizzazione ordinata per LPIPS
Modifica `DISPLAY_FROM` e `DISPLAY_N` e ri-esegui le due celle sotto
per navigare il ranking **senza ripetere l'inferenza**.

In [ ]:
DISPLAY_FROM = 0    # 0 = migliore LPIPS
DISPLAY_N    = 50   # numero di righe da mostrare

In [ ]:
# Carica da disco se sorted_results non è in memoria (es. kernel riavviato)
try:
    sorted_results
except NameError:
    with open(os.path.join(RESULTS_DIR, "results.json")) as f:
        sorted_results = sorted(json.load(f), key=lambda r: r["lpips"])
    print(f"Caricati {len(sorted_results)} risultati da disco")

page = sorted_results[DISPLAY_FROM : DISPLAY_FROM + DISPLAY_N]
n = len(page)

best  = sorted_results[DISPLAY_FROM]["lpips"]
worst = sorted_results[min(DISPLAY_FROM + DISPLAY_N - 1, len(sorted_results) - 1)]["lpips"]
print(f"Rank #{DISPLAY_FROM + 1} → #{DISPLAY_FROM + n}  |  LPIPS {best:.4f} — {worst:.4f}")

fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for i, r in enumerate(page):
    rank = DISPLAY_FROM + i + 1
    axes[i, 0].imshow(Image.open(r["ground_path"]).convert("RGB"))
    axes[i, 1].imshow(Image.open(r["gen_path"]).convert("RGB"))
    axes[i, 2].imshow(Image.open(r["sat_path"]).convert("RGB"))
    for ax in axes[i]:
        ax.axis("off")
    axes[i, 0].set_title(f"#{rank}  Ground input", fontsize=9)
    axes[i, 1].set_title(f"Generated  (LPIPS {r['lpips']:.4f})", fontsize=9, color="steelblue")
    axes[i, 2].set_title("Satellite GT", fontsize=9, color="green")

plt.tight_layout()
plt.show()